In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from vinishko.pipeline.steps.normalization.seg import open_image
from vinishko.pipeline.steps.normalization.normalize import NormalizationReason, Normalizer, load_config
from vinishko.pipeline.structs import Candidate, Rejection
from pathlib import Path
import random
import plotly.graph_objects as go
from PIL import Image
from plotly.colors import hex_to_rgb, qualitative
from transformers.models.dinov3_vit import DINOv3ViTModel
import os 
from plotly.subplots import make_subplots

In [3]:
img_paths = list(Path("/home/pomelk1n/Workspace/vinishko/datasets/visual-encoder/winesensed/images").iterdir())

In [4]:
config_path = Path("/home/pomelk1n/Workspace/vinishko/vinishko/pipeline/steps/normalization/normalize.toml")

normalizer = Normalizer(load_config(config_path, overrides=[]), config_path.parent)

In [5]:
REJECTED, LABEL_OK, LABEL_BAD = "#9aa0a6", "#ffd60a", "#ff375f"


def mask_trace(polys, name, color, hover, alpha=0.35, visible=True):
    """Маска заливкой с контуром; отдельная строка легенды, клик по ней скрывает маску."""
    xs, ys = [], []
    for p in polys:
        if len(p) >= 3:
            xs += [*(pt[0] for pt in p), p[0][0], None]
            ys += [*(pt[1] for pt in p), p[0][1], None]
    r, g, b = hex_to_rgb(color)
    return go.Scatter(
        x=xs, y=ys, name=name, mode="lines", fill="toself", fillcolor=f"rgba({r},{g},{b},{alpha})",
        line={"color": color, "width": 2}, hoveron="fills", hoverinfo="text", text=hover,
        visible=True if visible else "legendonly",
    )


def item_traces(item, n):
    """Следы одной бутылки: годная — цветная маска и жёлтая этикетка; отказ — серая маска и красная этикетка, если она нашлась."""
    if isinstance(item, Candidate):
        color = qualitative.Plotly[(item.index - 1) % len(qualitative.Plotly)]
        hover = f"годная бутылка {item.index}<br>score {item.score}<br>угол {item.angle}°<br>uuid {item.uuid[:8]}"
        return [
            mask_trace(item.bottle, f"b{item.index} бутылка", color, hover),
            mask_trace(item.label, f"b{item.index} этикетка", LABEL_OK, f"b{item.index} основная этикетка"),
        ]
    hover = f"{item.message}<br>score {item.score}<br><i>{item.reason.description}</i>"
    shown = item.reason != NormalizationReason.NOT_TARGET  # фоновые бутылки выключены до клика, отказы по этикетке видны сразу
    traces = [mask_trace(item.bottle, f"отказ {n} · {item.reason.title}", REJECTED, hover, visible=shown)]
    if item.label:
        traces.append(mask_trace(item.label, f"отказ {n} · этикетка", LABEL_BAD, hover, visible=shown))
    return traces


def add_picture(fig, picture, col):
    """Картинка фоном своей колонки: оси в её пикселях, y вниз, пропорции сохраняются."""
    w, h = picture.size
    ref = "" if col == 1 else str(col)
    fig.add_layout_image(source=picture, x=0, y=0, sizex=w, sizey=h, xref=f"x{ref}", yref=f"y{ref}", xanchor="left", yanchor="top", sizing="stretch", layer="below")
    fig.update_xaxes(range=[0, w], visible=False, constrain="domain", row=1, col=col)
    fig.update_yaxes(range=[h, 0], visible=False, scaleanchor=f"x{ref}", scaleratio=1, constrain="domain", row=1, col=col)


def plot_normalization(image: Image.Image, samples: list | None, items: list, size: tuple[int, int] = (1200, 800)) -> go.Figure:
    """Слева оригинал с масками всех найденных бутылок, справа кропы, которые идут дальше по пайплайну; size — ширина и высота графика.

    samples и items — пара, которую отдаёт normalizer(image). Если samples равен None, годных бутылок нет и справа пусто.
    Клик по строке легенды скрывает или показывает маску, двойной клик оставляет одну, кнопки сверху переключают все разом.
    """
    crops = [Image.fromarray(s.crop) for s in samples or []]
    pictures = [image, *crops]
    titles = ["оригинал", *(f"b{s.candidate.index} · в пайплайн · {s.uuid[:8]}" for s in samples or [])]
    widths = [p.width / p.height for p in pictures]  # при общей высоте ширина колонки пропорциональна отношению сторон
    fig = make_subplots(rows=1, cols=len(pictures), column_widths=widths, subplot_titles=titles, horizontal_spacing=0.02)
    for col, picture in enumerate(pictures, 1):
        add_picture(fig, picture, col)
    for n, item in enumerate(items, 1):
        fig.add_traces(item_traces(item, n), rows=1, cols=1)
    buttons = [
        {"label": "все маски", "method": "restyle", "args": [{"visible": True}]},
        {"label": "только фото", "method": "restyle", "args": [{"visible": "legendonly"}]},
    ]
    n_ok = sum(isinstance(item, Candidate) for item in items)
    fig.update_layout(
        width=size[0], height=size[1],
        title={"text": f"годных {n_ok} · отказов {len(items) - n_ok}" + ("" if samples else " · в пайплайн ничего не идёт"), "y": 0.99, "yanchor": "top"},
        margin={"l": 10, "r": 10, "t": 120, "b": 10},
        legend={"itemclick": "toggle", "itemdoubleclick": "toggleothers"},
        updatemenus=[{"type": "buttons", "direction": "right", "x": 0, "xanchor": "left", "y": 1.06, "yanchor": "bottom", "buttons": buttons}],
        hoverlabel={"align": "left"},
    )
    return fig

In [32]:
path = random.choice(img_paths)
#path = Path("/home/pomelk1n/Workspace/vinishko/datasets/visual-encoder/off/images/8411767281144.jpg")
image = open_image(str(path))
samples, items = normalizer(image)
plot_normalization(image, samples, items, size=(1200, 800))

In [33]:
path

PosixPath('/home/pomelk1n/Workspace/vinishko/datasets/visual-encoder/winesensed/images/MPkyJOnLQvuZBTHpUDrzrQ.jpg')

In [9]:
model = DINOv3ViTModel.from_pretrained("facebook/dinov3-vitl16-pretrain-lvd1689m", token=True)
model

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (model): DINOv3ViTEncoder(
    (layer): ModuleList(
      (0-23): 24 x DINOv3ViTLayer(
        (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (attention): DINOv3ViTAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (layer_scale1): DINOv3ViTLayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): DINOv3ViTMLP(
          (up_proj): Linear(in_features=1024, out_features=4096, bias=True)
          (down_proj

In [15]:
items

[Rejection(reason=<NormalizationReason.LABEL_BLURRY: 'label_blurry'>, detail='резкость 206, нужно от 300', score=0.8755, bottle=[[[23, 0], [19, 3], [18, 6], [19, 59], [21, 69], [19, 118], [19, 174], [21, 199], [21, 237], [23, 246], [22, 254], [25, 319], [24, 335], [26, 353], [25, 376], [27, 387], [26, 405], [27, 418], [29, 424], [28, 449], [30, 465], [30, 510], [32, 524], [33, 557], [37, 581], [42, 596], [52, 614], [74, 637], [100, 637], [102, 639], [108, 637], [115, 639], [123, 637], [135, 639], [139, 637], [224, 637], [229, 639], [237, 637], [248, 639], [253, 637], [274, 637], [276, 639], [282, 637], [289, 639], [293, 637], [296, 639], [306, 639], [314, 637], [334, 637], [359, 639], [362, 637], [392, 637], [407, 632], [420, 623], [421, 620], [430, 610], [432, 604], [437, 597], [437, 594], [440, 590], [440, 585], [444, 578], [448, 548], [448, 495], [450, 485], [450, 438], [448, 434], [448, 413], [450, 412], [450, 385], [447, 381], [448, 367], [445, 365], [447, 361], [447, 348], [445, 

In [20]:
items[0].label

[[[54, 113],
  [51, 116],
  [51, 163],
  [57, 411],
  [59, 439],
  [59, 507],
  [61, 515],
  [61, 533],
  [68, 538],
  [91, 546],
  [117, 553],
  [134, 555],
  [138, 557],
  [192, 563],
  [252, 564],
  [272, 562],
  [294, 562],
  [363, 552],
  [398, 542],
  [425, 531],
  [428, 526],
  [428, 441],
  [430, 435],
  [429, 396],
  [431, 329],
  [430, 310],
  [432, 162],
  [431, 116],
  [420, 111],
  [372, 101],
  [346, 99],
  [338, 97],
  [305, 96],
  [290, 94],
  [202, 94],
  [149, 97],
  [143, 99],
  [131, 99],
  [101, 104],
  [93, 104]]]

In [9]:
image.size

(480, 640)